In [1]:
q1 = 'i just discovered the course, can i still join?'
q2 = 'i just found out about the program, can i still enroll?'

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
print(model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
v1

array([-7.94801954e-03, -9.18932483e-02, -1.14074098e-02,  2.18466446e-02,
        1.11858072e-02, -1.30818337e-02, -7.39962384e-02, -9.87467393e-02,
       -1.05911404e-01, -3.03381961e-02, -2.92174350e-02,  2.33883206e-02,
        7.87485484e-03,  4.15800251e-02,  2.42720619e-02, -3.65723670e-02,
       -5.31095527e-02, -1.94868986e-02, -2.26979852e-02,  8.76781158e-03,
       -1.10785283e-01,  3.97906676e-02, -4.18480039e-02,  2.82960795e-02,
       -1.28302453e-02,  1.72567442e-02,  3.10428198e-02,  9.51102898e-02,
       -1.90717876e-02, -6.23235814e-02, -3.59101146e-02,  6.46180511e-02,
        3.06465030e-02,  2.39972137e-02,  2.06127614e-02,  9.87384189e-03,
        7.63835981e-02, -6.50510937e-02,  4.07932932e-03,  2.34527402e-02,
       -2.49407068e-02, -2.95020565e-02, -1.74879469e-02,  4.62139286e-02,
        2.48922799e-02,  1.08981758e-01, -5.67837432e-02, -6.95324540e-02,
        4.35404200e-03,  2.85132304e-02,  2.75795199e-02, -2.49843970e-02,
       -6.82142610e-03,  

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.39572883)

In [9]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [10]:
v2.dot(dv)

np.float32(0.019730438)

In [11]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [12]:
from ingest import load_faq_data

documents = load_faq_data()

In [13]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [14]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [15]:
len(texts)

1406

In [16]:
texts[1:50]

["Course: What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up front. The course covers Git and GitHub (see *How do I use Git/GitHub for this course?*), and you'll pick up the command-line/Linux basics you need during the setup modules.",
 "Course: Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 "Course: I have registered for the Data Engineering Bootca

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1406

In [18]:
vectors

[array([-2.67062243e-02, -1.22457556e-01,  1.59441810e-02,  6.09419961e-03,
        -1.11915041e-02, -6.55021220e-02, -7.62883872e-02, -2.08820179e-02,
        -9.27567407e-02,  3.55664827e-02,  3.15623023e-02, -1.09017007e-02,
        -2.45336294e-02, -1.82476453e-02,  3.43917273e-02, -1.32052284e-02,
         7.22677913e-03, -1.54126719e-01,  6.41842559e-02, -9.07449145e-03,
         3.94655168e-02, -2.99637597e-02,  2.08512675e-02,  3.71391624e-02,
         3.52526940e-02, -2.49497825e-03,  7.71128982e-02,  2.78048050e-02,
         1.54832490e-02,  4.92821680e-03,  1.48516195e-03,  3.93927470e-02,
         7.25188628e-02,  9.02841091e-02,  5.25653325e-02,  2.72272304e-02,
         3.86085585e-02, -7.50263035e-02, -2.48754397e-02,  1.06018260e-01,
        -4.82870899e-02, -5.00598513e-02, -4.16486375e-02,  7.91618153e-02,
         5.06461747e-02, -3.68613109e-04, -2.83320853e-03, -2.59598717e-02,
        -3.82818393e-02,  8.59545469e-02, -2.85155009e-02, -7.23346546e-02,
        -2.0

In [19]:
vectors[10].shape

(384,)

In [20]:
import numpy as np
X = np.array(vectors)

In [21]:
X.shape

(1406, 384)

In [22]:
scores = X.dot(v1)

In [23]:
scores

array([0.4130352 , 0.26119855, 0.6088087 , ..., 0.23222183, 0.2057406 ,
       0.14642397], shape=(1406,), dtype=float32)

In [24]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(1009), np.float32(0.831779))

In [25]:
documents[1009]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [26]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [27]:
scores[top5]

array([0.831779  , 0.6845695 , 0.61755615, 0.6088087 , 0.58479655],
      dtype=float32)

In [28]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.831779
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6845695
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.61755615
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'se

In [29]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [30]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [31]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [32]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [33]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [34]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [35]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)  #this gives text search result

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [36]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder  #it is our model

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query) #this turns query into a vector
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,             #it uses the query vector not the original text documents
            num_results=num_results,
            filter_dict=filter_dict
        )

In [37]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [38]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)  #this is doing vector search instead of text search.

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

uptill now we have used minsearch, now we will use sqlitesearch

### Vector search with Sqlitesearch

why? Because under the hood minsearch is doing exactly what we did above steps. It is not ideal for large documents

We will use ANN, approximate nearest neighbour, instead of NN, nearest neighbours approch. ANN is used for scale.

NN compute similarity between vector and all documents, and give exact top 5 or whatever. ANN give approximate, not exactly top 5, but close enough, results will make sense. It is way faster, we dont need to check our vector with every document.

In [45]:
vs_index.clear()

In [46]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", #IVF stands for Inverted File Index — it's an approximate nearest neighbor (ANN) search strategy
    db_path="faq_vectors2.db"
) 

it will use persistent database

In [47]:
vs_index.fit(vectors, documents)

In [48]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query) #we always have to encode query before we perfom search

results = vs_index.search(query_vector, num_results=5)

In [49]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

##### filtering by course

In [50]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [51]:
vs_index.close()

In a new Python session, you can reopen the index without re-computing embeddings, see vector_search_persistent.ipynb